In [ ]:

from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive')

print("=== TOP-LEVEL DRIVE FOLDERS ===")
print(*os.listdir(ROOT)[:50], sep="\n")

print("\n=== POSSIBLE SIGNAL DATA FILES ===")
extensions = {".csv", ".npy", ".npz", ".mat",
              ".hea", ".dat", ".parquet", ".h5"}

count = 0
for folder, dirs, files in os.walk(ROOT):
    depth = len(Path(folder).relative_to(ROOT).parts)
    if depth > 4:
        dirs[:] = []
        continue

    for filename in files:
        p = Path(folder) / filename
        if p.suffix.lower() in extensions:
            print(p, "|", round(p.stat().st_size/1e6, 2), "MB")
            count += 1
            if count >= 40:
                break

    if count >= 40:
        break

print("\nFiles displayed:", count)


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path("/content/drive/MyDrive/SentinelWard-work/data-audit")

# Select the most recent audit folder
audits = sorted(
    [p for p in BASE.iterdir()
     if p.is_dir() and p.name[:8].isdigit()]
)
AUDIT = audits[-1]
print("AUDIT FOLDER:", AUDIT.name)

# Inspect important metadata files
names = [
    "benchmark_data_split.csv",
    "event_labels.csv",
    "vtac_development_header_manifest.csv",
    "challenge2015_labels.csv",
    "challenge2011_quality_labels.csv"
]

for name in names:
    p = AUDIT / name
    if not p.exists():
        continue

    df = pd.read_csv(p)
    print(f"\n{'='*55}")
    print("FILE:", name)
    print("SHAPE:", df.shape)
    print("COLUMNS:", df.columns.tolist())
    print("FIRST TWO ROWS:")
    print(df.head(2).to_string(index=False, max_cols=12))

# Inspect cached signal arrays without loading everything
cache = AUDIT / "vtac_native250_cache_v1"
batches = sorted(cache.glob("*.npz"))

print(f"\n{'='*55}")
print("CACHED BATCH COUNT:", len(batches))

if batches:
    with np.load(batches[0], allow_pickle=False) as z:
        print("FIRST BATCH:", batches[0].name)
        for key in z.files:
            a = z[key]
            print(f"{key}: shape={a.shape}, dtype={a.dtype}")

# Show top-level project folders
project = BASE.parent
print("\nPROJECT FOLDERS:")
for p in sorted(project.iterdir()):
    print("[DIR]" if p.is_dir() else "[FILE]", p.name)


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

# Continue using AUDIT from Step 2
cache = AUDIT / "vtac_native250_cache_v1"
files = sorted(cache.glob("batch_*.npz"))

assert files, "No cached signal batches found!"

parts = {k: [] for k in
         ["x", "valid", "y", "event_id", "record_id", "split"]}

for file in files:
    with np.load(file, allow_pickle=False) as z:
        for key in parts:
            parts[key].append(z[key])

X = np.concatenate(parts["x"])
MASK = np.concatenate(parts["valid"])
Y_VT = np.concatenate(parts["y"])

df = pd.DataFrame({
    "event_id": np.concatenate(parts["event_id"]),
    "record_id": np.concatenate(parts["record_id"]),
    "split": np.concatenate(parts["split"]),
    "vt_label": Y_VT
})

assert X.ndim == 3 and X.shape[1:] == (2, 2500)
assert MASK.shape == X.shape
assert len(df) == len(X)

print("=== SIGNAL DATA ===")
print("Total windows:", len(X))
print("Signal shape:", X.shape)
print("Memory (MB):", round(
    (X.nbytes + MASK.nbytes) / 1e6, 2
))

print("\n=== SPLIT DISTRIBUTION ===")
print(df["split"].value_counts().to_string())

print("\n=== UNIQUE RECORDS ===")
print(df.groupby("split")["record_id"].nunique())

# Detect record-level train/test leakage
record_splits = df.groupby("record_id")["split"].nunique()
overlap = record_splits[record_splits > 1]
print("\nRecords appearing in multiple splits:", len(overlap))
if len(overlap):
    print("WARNING: Existing splits overlap by record!")

print("\n=== DATA VALIDITY ===")
valid_fraction = MASK.mean(axis=(1, 2))
print("Median valid fraction:",
      round(float(np.median(valid_fraction)), 4))
print("Windows >= 90% valid:",
      int((valid_fraction >= 0.9).sum()))

bad = (~np.isfinite(X)) & MASK
print("Non-finite valid samples:", int(bad.sum()))

print("\n=== REPEATED RECORDS ===")
counts = df.groupby("record_id").size()
print("Unique records:", len(counts))
print("Records with >= 2 windows:", int((counts >= 2).sum()))
print("Records with >= 5 windows:", int((counts >= 5).sum()))

print("\n=== VT LABELS (NOT REPLAY LABELS) ===")
print(df["vt_label"].value_counts().to_string())

print("\n=== CHANNEL METADATA ===")
manifest = pd.read_csv(
    AUDIT / "vtac_development_header_manifest.csv"
)
cols = [
    "event_id", "sampling_rate_hz",
    "channel_names", "preferred_ecg",
    "has_pleth_header", "ecg_and_pleth_header"
]
print(manifest[cols].head(5).to_string(index=False))

print("\n=== CACHE / MANIFEST MATCH ===")
matched = df["event_id"].isin(manifest["event_id"])
print("Matched events:", int(matched.sum()))
print("Unmatched events:", int((~matched).sum()))

print("\nSTEP 3 INSPECTION COMPLETE")


In [ ]:

from sklearn.model_selection import GroupShuffleSplit
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# Require >=90% valid samples in EACH channel.
eligible = (MASK.mean(axis=2) >= 0.90).all(axis=1)

def pair_features(a, b, mask_a, mask_b):
    features = {}
    for ch in range(2):
        valid = mask_a[ch] & mask_b[ch]
        u = a[ch, valid].astype(np.float64)
        v = b[ch, valid].astype(np.float64)

        su = max(np.std(u), 1e-6)
        sv = max(np.std(v), 1e-6)
        zu = (u - u.mean()) / su
        zv = (v - v.mean()) / sv

        prefix = f"ch{ch}_"
        features[prefix+"corr"] = float(
            np.clip(np.mean(zu*zv), -1, 1))
        features[prefix+"nrmse"] = float(
            np.sqrt(np.mean((u-v)**2))/su)
        features[prefix+"shape_rmse"] = float(
            np.sqrt(np.mean((zu-zv)**2)))
        features[prefix+"diff_rmse"] = float(
            np.sqrt(np.mean(np.diff(u-v)**2)) /
            max(np.std(np.diff(u)), 1e-6))

        # Minimum mismatch across 2-second segments.
        segment_errors = []
        for start in range(0, 2500, 500):
            m = mask_a[ch, start:start+500] & \
                mask_b[ch, start:start+500]
            if m.sum() < 300:
                continue
            aa = a[ch, start:start+500][m]
            bb = b[ch, start:start+500][m]
            segment_errors.append(
                np.sqrt(np.mean((aa-bb)**2))/su)

        features[prefix+"best_segment"] = float(
            min(segment_errors)) if segment_errors else 1e6

    return features

rows = []

# Group by source split and record to avoid cross-record mixing.
groups = df.loc[eligible].groupby(
    ["split", "record_id"]
).groups

for (source_split, record_id), indices in groups.items():
    ids = np.asarray(list(indices), dtype=int)
    if len(ids) < 2:
        continue

    ids = rng.permutation(ids)[:3]

    for j, ref_idx in enumerate(ids):
        other_idx = ids[(j+1) % len(ids)]

        ref = X[ref_idx]
        ref_mask = MASK[ref_idx]

        # Class 0: Different genuine recorded event.
        normal_features = pair_features(
            ref, X[other_idx],
            ref_mask, MASK[other_idx])

        rows.append({
            **normal_features,
            "y4": 0,
            "variant": "genuine",
            "record_id": record_id,
            "source_split": source_split
        })

        # Class 1: Synthetic replay of the reference.
        if j % 2 == 0:
            replay = ref.copy()
            variant = "exact"
        else:
            stds = np.array([
                np.std(ref[ch, ref_mask[ch]])
                for ch in range(2)
            ], dtype=np.float32)[:, None]

            gain = rng.uniform(0.98, 1.02, (2, 1))
            offset = rng.uniform(-0.01, 0.01, (2, 1))
            noise_level = rng.uniform(0.005, 0.03)

            replay = (
                ref * gain +
                stds * offset +
                rng.normal(size=ref.shape) *
                stds * noise_level
            ).astype(np.float32)
            variant = "modified"

        replay_features = pair_features(
            ref, replay, ref_mask, ref_mask)

        rows.append({
            **replay_features,
            "y4": 1,
            "variant": variant,
            "record_id": record_id,
            "source_split": source_split
        })

pairs = pd.DataFrame(rows)
feature_cols = [
    c for c in pairs.columns if c.startswith("ch")
]

print("=== MODEL 4 TRAINING EXAMPLES ===")
print("Total pairs:", len(pairs))
print("\nLabels:")
print(pairs.y4.value_counts())
print("\nSource partitions:")
print(pd.crosstab(pairs.source_split, pairs.y4))
print("\nReplay variants:")
print(pairs.variant.value_counts())
print("\nFeatures:", feature_cols)
print("Finite features:",
      np.isfinite(pairs[feature_cols].to_numpy()).all())

# Detect ambiguous genuine pairs which are exact duplicates.
ambiguous = pairs[
    (pairs.y4 == 0) &
    (pairs.ch0_nrmse < 1e-6) &
    (pairs.ch1_nrmse < 1e-6)
]
print("\nExact-duplicate genuine pairs:", len(ambiguous))

assert len(pairs) > 0
assert len(pairs[pairs.source_split == "train"]) > 0
assert len(pairs[pairs.source_split == "validation"]) > 0
assert np.isfinite(pairs[feature_cols].to_numpy()).all()

print("\nSTEP 4 COMPLETE")


In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_recall_curve
)
from sklearn.model_selection import GroupShuffleSplit
from pathlib import Path
import joblib

# Keep original validation partition out of training.
pool = pairs[pairs.source_split == "train"].reset_index(drop=True)
holdout = pairs[pairs.source_split == "validation"].reset_index(drop=True)

splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.20, random_state=42
)
train_idx, dev_idx = next(splitter.split(
    pool, pool.y4, groups=pool.record_id
))

train = pool.iloc[train_idx]
dev = pool.iloc[dev_idx]

assert set(train.record_id).isdisjoint(dev.record_id)
assert set(pool.record_id).isdisjoint(holdout.record_id)

print("Training pairs:", len(train))
print("Development pairs:", len(dev))
print("Held-out pairs:", len(holdout))

# Train baseline model
model4 = RandomForestClassifier(
    n_estimators=250,
    max_depth=12,
    min_samples_leaf=3,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

model4.fit(train[feature_cols], train.y4)

# Select threshold using development data ONLY.
dev_proba = model4.predict_proba(
    dev[feature_cols])[:, 1]

precision, recall, thresholds = precision_recall_curve(
    dev.y4, dev_proba
)

f1 = (
    2 * precision[:-1] * recall[:-1] /
    np.maximum(precision[:-1] + recall[:-1], 1e-12)
)

threshold = float(thresholds[np.argmax(f1)])
print("\nSelected development threshold:", round(threshold, 4))

def evaluate(name, data, scores, threshold):
    truth = data.y4.to_numpy()
    pred = (scores >= threshold).astype(int)

    print("\n===", name, "===")
    print("Confusion matrix [TN FP; FN TP]:")
    print(confusion_matrix(truth, pred, labels=[0, 1]))
    print(classification_report(
        truth, pred, labels=[0, 1],
        target_names=["Genuine", "Replay"],
        zero_division=0
    ))
    if len(np.unique(truth)) == 2:
        print("ROC-AUC:", round(
            roc_auc_score(truth, scores), 4))

    for variant in ["exact", "modified"]:
        m = (data.variant.to_numpy() == variant)
        if m.any():
            print(f"{variant} replay recall:",
                  round(float(pred[m].mean()), 4))

evaluate("INTERNAL DEVELOPMENT", dev,
         dev_proba, threshold)

holdout_proba = model4.predict_proba(
    holdout[feature_cols])[:, 1]
evaluate("HELD-OUT VALIDATION", holdout,
         holdout_proba, threshold)

# Compare against a simple exact duplicate detector.
exact_match_pred = (
    (holdout.ch0_nrmse < 1e-6) &
    (holdout.ch1_nrmse < 1e-6)
).astype(int).to_numpy()

print("\n=== EXACT-MATCH BASELINE ===")
print(classification_report(
    holdout.y4, exact_match_pred,
    labels=[0, 1],
    target_names=["Genuine", "Replay"],
    zero_division=0
))

importance = pd.Series(
    model4.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\n=== FEATURE IMPORTANCE ===")
print(importance.to_string())

# Export model and preprocessing settings
OUT = Path("/content/drive/MyDrive/SentinelWard-work/m4")
OUT.mkdir(parents=True, exist_ok=True)

model_path = OUT / "model4_replay_baseline.joblib"
joblib.dump({
    "model": model4,
    "feature_cols": feature_cols,
    "threshold": threshold,
    "sampling_rate_hz": 250,
    "window_samples": 2500,
    "minimum_channel_valid_fraction": 0.90,
    "training_type": "synthetic_pairwise_replay",
    "channel_mapping_verified": False
}, model_path)

print("\nMODEL SAVED:", model_path)
print("STEP 5 COMPLETE")


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

rng = np.random.default_rng(2026)
stress_rows = []

assert len(X) == len(df) == len(MASK)
assert len(feature_cols) == 10

# Use existing held-out validation records only.
val_groups = df.loc[
    (df["split"] == "validation") & eligible
].groupby("record_id").groups

def add_case(name, ref, candidate, ref_mask,
             candidate_mask, record_id, label):
    f = pair_features(
        ref, candidate, ref_mask, candidate_mask
    )
    stress_rows.append({
        **f,
        "scenario": name,
        "record_id": record_id,
        "expected_replay": label
    })

for record_id, group_indices in val_groups.items():
    ids = np.asarray(list(group_indices), dtype=int)

    if len(ids) < 2:
        continue

    # Two distinct recorded events from the same record.
    ref_idx, current_idx = rng.choice(
        ids, size=2, replace=False
    )

    ref = X[ref_idx]
    current = X[current_idx]
    rm = MASK[ref_idx]
    cm = MASK[current_idx]

    # Genuine comparison
    add_case("genuine", ref, current, rm, cm,
             record_id, 0)

    # Partial replay in both channels
    for seconds in [1, 3, 5]:
        n = seconds * 250
        start = int(rng.integers(0, 2500 - n + 1))
        end = start + n

        modified = current.copy()
        modified_mask = cm.copy()

        modified[:, start:end] = ref[:, start:end]
        modified_mask[:, start:end] = rm[:, start:end]

        add_case(
            f"partial_{seconds}s",
            ref, modified, rm, modified_mask,
            record_id, 1
        )

    # Replay only one signal channel
    for ch in range(2):
        n = 750  # 3 seconds
        start = int(rng.integers(0, 2500 - n + 1))
        end = start + n

        modified = current.copy()
        modified_mask = cm.copy()

        modified[ch, start:end] = ref[ch, start:end]
        modified_mask[ch, start:end] = rm[ch, start:end]

        add_case(
            f"single_channel_{ch}",
            ref, modified, rm, modified_mask,
            record_id, 1
        )

    # Full replay with stronger modifications
    stds = np.array([
        np.std(ref[ch, rm[ch]])
        for ch in range(2)
    ], dtype=np.float32)[:, None]

    gain = rng.uniform(0.95, 1.05, (2, 1))
    noise = rng.normal(size=ref.shape) * stds * 0.08

    modified = (ref * gain + noise).astype(np.float32)

    add_case(
        "modified_full", ref, modified,
        rm, rm, record_id, 1
    )

stress = pd.DataFrame(stress_rows)

assert len(stress) > 0
assert np.isfinite(stress[feature_cols].to_numpy()).all()

# Evaluate with the SAME trained model and threshold.
stress["replay_score"] = model4.predict_proba(
    stress[feature_cols]
)[:, 1]

stress["predicted_replay"] = (
    stress["replay_score"] >= threshold
).astype(int)

report = stress.groupby(
    ["scenario", "expected_replay"],
    as_index=False
).agg(
    cases=("predicted_replay", "size"),
    flagged_rate=("predicted_replay", "mean"),
    mean_score=("replay_score", "mean")
)

report["flagged_percent"] = (
    report["flagged_rate"] * 100
).round(2)

report["metric"] = np.where(
    report["expected_replay"] == 1,
    "Replay detection rate",
    "False-positive rate"
)

print("=== MODEL 4 ROBUSTNESS REPORT ===")
print(report[
    ["scenario", "cases", "metric",
     "flagged_percent", "mean_score"]
].to_string(index=False))

print("\n=== OVERALL TEST ===")
print("Total stress-test cases:", len(stress))
print("Records tested:", stress.record_id.nunique())
print("Frozen threshold:", threshold)

# Save separately: do not overwrite the baseline model.
OUT = Path("/content/drive/MyDrive/SentinelWard-work/m4")
OUT.mkdir(parents=True, exist_ok=True)

stress.to_csv(OUT / "stress_predictions_v1.csv", index=False)
report.to_csv(OUT / "stress_summary_v1.csv", index=False)

print("\nSaved stress-test results.")
print("STEP 6 COMPLETE")


In [ ]:

import numpy as np
import pandas as pd

rng2 = np.random.default_rng(2026)

def rolling_min_error(squared_error, valid, length, stride):
    """Lowest normalized RMSE over valid sliding windows."""
    values = np.where(valid, squared_error, 0.0)

    sums = np.concatenate(([0.0],
        np.cumsum(values, dtype=np.float64)))
    counts = np.concatenate(([0],
        np.cumsum(valid, dtype=np.int32)))

    starts = np.arange(
        0, len(values)-length+1, stride
    )
    ends = starts + length

    n = counts[ends] - counts[starts]
    mse = (sums[ends] - sums[starts]) / np.maximum(n, 1)

    errors = np.sqrt(np.maximum(mse, 0))
    errors[n < length*0.90] = np.inf

    return float(np.min(errors)) if np.isfinite(errors).any() else 1e3


def features_v2(ref, candidate, ref_mask, candidate_mask):
    # Preserve the 10 original global features.
    result = pair_features(
        ref, candidate, ref_mask, candidate_mask
    )

    for ch in range(2):
        valid = ref_mask[ch] & candidate_mask[ch]
        a = ref[ch].astype(np.float64)
        b = candidate[ch].astype(np.float64)

        mean_a = a[valid].mean()
        mean_b = b[valid].mean()
        std_a = max(a[valid].std(), 1e-4)
        std_b = max(b[valid].std(), 1e-4)

        # Amplitude-sensitive differences.
        raw_error = ((a-b)/std_a)**2

        # Shape differences after normalization.
        shape_error = (
            (a-mean_a)/std_a -
            (b-mean_b)/std_b
        )**2

        windows = [
            ("0p5s", 125, 25),
            ("1s",   250, 25),
            ("3s",   750, 125),
            ("5s",  1250, 250)
        ]

        for name, length, stride in windows:
            result[f"ch{ch}_raw_{name}"] = rolling_min_error(
                raw_error, valid, length, stride
            )
            result[f"ch{ch}_shape_{name}"] = rolling_min_error(
                shape_error, valid, length, stride
            )

    return result


ATTACKS = [
    "full_exact",
    "full_modified",
    "partial_1s",
    "partial_3s",
    "partial_5s",
    "single_channel_0",
    "single_channel_1",
    "partial_1s_noisy"
]

def simulate_attack(ref, current, rm, cm, mode):
    if mode == "full_exact":
        return ref.copy(), rm.copy()

    if mode == "full_modified":
        stds = np.array([
            max(np.std(ref[ch, rm[ch]]), 1e-4)
            for ch in range(2)
        ])[:, None]

        gain = rng2.uniform(0.95, 1.05, (2, 1))
        noise = rng2.normal(size=ref.shape) * stds * 0.08
        return (ref*gain + noise).astype(np.float32), rm.copy()

    # Partial or single-channel synthetic replay.
    length = {
        "partial_1s": 250,
        "partial_3s": 750,
        "partial_5s": 1250,
        "single_channel_0": 750,
        "single_channel_1": 750,
        "partial_1s_noisy": 250
    }[mode]

    start = int(rng2.integers(0, 2500-length+1))
    end = start + length

    out = current.copy()
    out_mask = cm.copy()

    channels = [0, 1]
    if mode == "single_channel_0":
        channels = [0]
    elif mode == "single_channel_1":
        channels = [1]

    for ch in channels:
        segment = ref[ch, start:end].copy()

        if mode == "partial_1s_noisy":
            scale = max(np.std(ref[ch, rm[ch]]), 1e-4)
            segment += rng2.normal(
                0, 0.015*scale, length
            )

        out[ch, start:end] = segment
        out_mask[ch, start:end] = rm[ch, start:end]

    return out, out_mask


# Only use records from the original TRAIN partition.
good = (
    df["split"].eq("train").to_numpy() &
    (MASK.mean(axis=2) >= 0.90).all(axis=1)
)

record_groups = df.loc[good].groupby("record_id").groups
record_groups = {
    k: np.asarray(list(v), dtype=int)
    for k, v in record_groups.items()
    if len(v) >= 2
}

records = np.array(sorted(record_groups))
rng2.shuffle(records)

n = len(records)
n_fit = int(0.70*n)
n_dev = int(0.85*n)

record_partitions = {
    "fit": records[:n_fit],
    "dev": records[n_fit:n_dev],
    "test": records[n_dev:]
}

assert all(len(v) > 0 for v in record_partitions.values())

rows_v2 = []

for partition, record_ids in record_partitions.items():
    for record_id in record_ids:
        ids = record_groups[record_id]
        a_idx, b_idx = rng2.choice(ids, size=2, replace=False)

        ref, current = X[a_idx], X[b_idx]
        rm, cm = MASK[a_idx], MASK[b_idx]

        # Class 0: Distinct recorded event.
        rows_v2.append({
            **features_v2(ref, current, rm, cm),
            "label": 0,
            "scenario": "genuine",
            "record_id": record_id,
            "partition": partition
        })

        # Train on sampled attack varieties.
        # Evaluate on all varieties in dev and test.
        modes = (
            rng2.choice(ATTACKS, size=3, replace=False)
            if partition == "fit" else ATTACKS
        )

        for mode in modes:
            altered, altered_mask = simulate_attack(
                ref, current, rm, cm, mode
            )

            rows_v2.append({
                **features_v2(
                    ref, altered, rm, altered_mask
                ),
                "label": 1,
                "scenario": mode,
                "record_id": record_id,
                "partition": partition
            })

data_v2 = pd.DataFrame(rows_v2)

feature_cols_v2 = [
    c for c in data_v2.columns
    if c.startswith("ch")
]

assert np.isfinite(
    data_v2[feature_cols_v2].to_numpy()
).all()

# Check separation across record groups.
sets = [set(v) for v in record_partitions.values()]
assert not (sets[0] & sets[1])
assert not (sets[0] & sets[2])
assert not (sets[1] & sets[2])

print("=== MODEL 4 V2 DATASET ===")
print("Total records:", n)
print("Total examples:", len(data_v2))
print("Features:", len(feature_cols_v2))
print("\nPartition sizes:")
print(pd.crosstab(
    data_v2.partition, data_v2.label
))
print("\nAttack scenarios:")
print(data_v2.scenario.value_counts())
print("\nSTEP 7 COMPLETE")


In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

fit2 = data_v2[data_v2.partition == "fit"].copy()
dev2 = data_v2[data_v2.partition == "dev"].copy()
test2 = data_v2[data_v2.partition == "test"].copy()

model4_v2 = RandomForestClassifier(
    n_estimators=300,
    max_depth=14,
    min_samples_leaf=3,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

model4_v2.fit(
    fit2[feature_cols_v2],
    fit2.label
)

# Select the threshold from genuine DEVELOPMENT examples.
dev2["score"] = model4_v2.predict_proba(
    dev2[feature_cols_v2]
)[:, 1]

negative_scores = dev2.loc[
    dev2.label == 0, "score"
].to_numpy()

# Target roughly <=1% development false-positive rate.
# This is an experimental choice, not a validated guarantee.
threshold_v2 = float(
    min(1.000001, np.quantile(negative_scores, 0.99) + 1e-6)
)

print("Selected threshold:", round(threshold_v2, 5))


def report_v2(data, title):
    data = data.copy()
    data["score"] = model4_v2.predict_proba(
        data[feature_cols_v2]
    )[:, 1]
    data["prediction"] = (
        data.score >= threshold_v2
    ).astype(int)

    print(f"\n=== {title} ===")
    print("Confusion matrix:")
    print(confusion_matrix(
        data.label, data.prediction,
        labels=[0, 1]
    ))

    print(classification_report(
        data.label, data.prediction,
        labels=[0, 1],
        target_names=["Genuine", "Simulated replay"],
        zero_division=0
    ))

    summary = data.groupby(
        ["scenario", "label"], as_index=False
    ).agg(
        cases=("prediction", "size"),
        flagged=("prediction", "sum"),
        mean_score=("score", "mean")
    )

    summary["flagged_percent"] = (
        100 * summary.flagged / summary.cases
    ).round(2)

    summary["metric"] = np.where(
        summary.label == 0,
        "False-positive rate",
        "Detection rate"
    )

    print(summary[
        ["scenario", "cases", "metric",
         "flagged_percent", "mean_score"]
    ].to_string(index=False))

    return data, summary


dev_predictions, dev_summary = report_v2(
    dev2, "DEVELOPMENT"
)

test_predictions, test_summary = report_v2(
    test2, "INTERNAL HELD-OUT TEST"
)

print("\n=== TOP FEATURES ===")
importance_v2 = pd.Series(
    model4_v2.feature_importances_,
    index=feature_cols_v2
).sort_values(ascending=False)

print(importance_v2.head(12).to_string())

# Save Version 2 separately from the baseline.
OUT = Path("/content/drive/MyDrive/SentinelWard-work/m4")
OUT.mkdir(parents=True, exist_ok=True)

joblib.dump({
    "model": model4_v2,
    "feature_columns": feature_cols_v2,
    "threshold": threshold_v2,
    "sampling_rate_hz": 250,
    "window_samples": 2500,
    "version": "m4_replay_v2",
    "experiment": "synthetic_aligned_replay",
    "patient_disjoint_verified": False
}, OUT / "model4_replay_v2.joblib")

test_predictions.to_csv(
    OUT / "v2_internal_test_predictions.csv",
    index=False
)
test_summary.to_csv(
    OUT / "v2_internal_test_summary.csv",
    index=False
)

print("\nMODEL 4 VERSION 2 SAVED")
print("STEP 8 COMPLETE")


In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
from pathlib import Path

rng3 = np.random.default_rng(20261016)
rows_v3 = []

# Original validation records, not used for V2 fitting.
valid_windows = (
    df["split"].eq("validation").to_numpy()
    & (MASK.mean(axis=2) >= 0.90).all(axis=1)
)

val_groups = df.loc[valid_windows].groupby("record_id").groups

assert not (
    set(df.loc[df.split == "train", "record_id"])
    & set(df.loc[df.split == "validation", "record_id"])
)

def add_test_case(name, label, record_id,
                  ref, candidate, rm, candidate_mask):
    feat = features_v2(
        ref, candidate, rm, candidate_mask
    )
    rows_v3.append({
        **feat,
        "scenario": name,
        "label": label,
        "record_id": record_id
    })

# Scenario, length, source start, destination start, channels, noise
attacks = [
    ("shifted_0p5s_both", 125, 250, 1800, [0, 1], False),
    ("shifted_1s_both", 250, 350, 1500, [0, 1], False),
    ("shifted_3s_both", 750, 1200, 150, [0, 1], False),
    ("shifted_1s_ch0", 250, 350, 1500, [0], False),
    ("shifted_1s_ch1", 250, 350, 1500, [1], False),
    ("shifted_1s_noisy", 250, 350, 1500, [0, 1], True),
]

for record_id, group in val_groups.items():
    ids = np.asarray(list(group), dtype=int)
    if len(ids) < 2:
        continue

    i, j = rng3.choice(ids, size=2, replace=False)
    ref, current = X[i], X[j]
    rm, cm = MASK[i], MASK[j]

    # Genuine recorded events
    add_test_case(
        "genuine", 0, record_id,
        ref, current, rm, cm
    )

    # Genuine with slight gain, noise and baseline drift
    std = np.array([
        max(np.std(current[ch, cm[ch]]), 1e-4)
        for ch in range(2)
    ], dtype=np.float32)[:, None]

    drift = np.linspace(-1, 1, 2500)[None, :]
    perturbed = (
        current * rng3.uniform(0.97, 1.03, (2, 1))
        + std * 0.02 * drift
        + rng3.normal(size=current.shape) * std * 0.01
    ).astype(np.float32)

    add_test_case(
        "genuine_mild_drift", 0, record_id,
        ref, perturbed, rm, cm
    )

    ref_std = np.array([
        max(np.std(ref[ch, rm[ch]]), 1e-4)
        for ch in range(2)
    ])

    for name, length, src, dst, channels, noisy in attacks:
        altered = current.copy()
        altered_mask = cm.copy()

        for ch in channels:
            segment = ref[ch, src:src+length].copy()

            if noisy:
                segment = (
                    segment * 1.02
                    + rng3.normal(
                        0, 0.02*ref_std[ch], length
                    )
                )

            altered[ch, dst:dst+length] = segment
            altered_mask[ch, dst:dst+length] = (
                rm[ch, src:src+length]
            )

        add_test_case(
            name, 1, record_id,
            ref, altered, rm, altered_mask
        )

challenge = pd.DataFrame(rows_v3)

assert len(challenge) > 0
assert np.isfinite(
    challenge[feature_cols_v2].to_numpy()
).all()

# Frozen V2 model, frozen V2 threshold.
challenge["score"] = model4_v2.predict_proba(
    challenge[feature_cols_v2]
)[:, 1]

challenge["prediction"] = (
    challenge.score >= threshold_v2
).astype(int)

summary = challenge.groupby(
    ["scenario", "label"], as_index=False
).agg(
    cases=("prediction", "size"),
    flagged=("prediction", "sum"),
    mean_score=("score", "mean")
)

summary["flagged_percent"] = (
    100 * summary.flagged / summary.cases
).round(2)

summary["metric"] = np.where(
    summary.label == 0,
    "False-positive rate",
    "Replay detection rate"
)

print("=== MODEL 4 V2 SHIFTED-REPLAY CHALLENGE ===")
print(summary[
    ["scenario", "cases", "metric",
     "flagged_percent", "mean_score"]
].to_string(index=False))

print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(
    challenge.label,
    challenge.prediction,
    labels=[0, 1]
))

print("\nRecords evaluated:", challenge.record_id.nunique())
print("Frozen threshold:", threshold_v2)

OUT = Path("/content/drive/MyDrive/SentinelWard-work/m4")
OUT.mkdir(parents=True, exist_ok=True)

challenge.to_csv(
    OUT / "v2_shifted_challenge_predictions.csv",
    index=False
)
summary.to_csv(
    OUT / "v2_shifted_challenge_summary.csv",
    index=False
)

print("\nSTEP 9 COMPLETE")


In [ ]:

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

rng4 = np.random.default_rng(4004)

# Downsample by selecting every fifth sample for this
# exploratory matching experiment: 250 Hz -> 50 Hz.
# Not intended for clinical signal processing.

def get_windows(signal, mask, length, stride=5):
    s = signal[::5]
    m = mask[::5]

    windows = sliding_window_view(s, length)[::stride]
    validity = sliding_window_view(m, length)[::stride]

    return windows, validity.all(axis=1)


def minimum_distance(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.nan

    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)

    aa = np.sum(a*a, axis=1)[:, None]
    bb = np.sum(b*b, axis=1)[None, :]
    cross = a @ b.T

    squared = np.maximum(
        (aa + bb - 2*cross) / a.shape[1], 0
    )

    return float(np.sqrt(squared.min()))


def lag_features(ref, candidate, rm, cm):
    output = {}

    means = np.array([
        np.mean(ref[ch, rm[ch]]) for ch in range(2)
    ])
    scales = np.array([
        max(np.std(ref[ch, rm[ch]]), 1e-4)
        for ch in range(2)
    ])

    for duration, width in [
        ("0p5s", 25),
        ("1s", 50),
        ("3s", 150)
    ]:
        aw, av, bw, bv = [], [], [], []

        for ch in range(2):
            a, ma = get_windows(
                ref[ch], rm[ch], width
            )
            b, mb = get_windows(
                candidate[ch], cm[ch], width
            )

            a = (a - means[ch]) / scales[ch]
            b = (b - means[ch]) / scales[ch]

            aw.append(a)
            av.append(ma)
            bw.append(b)
            bv.append(mb)

            output[f"lag_ch{ch}_{duration}"] = (
                minimum_distance(a[ma], b[mb])
            )

        # Both channels must match at the SAME
        # reference and candidate positions.
        joint_a_valid = av[0] & av[1]
        joint_b_valid = bv[0] & bv[1]

        joint_a = np.concatenate([
            aw[0][joint_a_valid],
            aw[1][joint_a_valid]
        ], axis=1)

        joint_b = np.concatenate([
            bw[0][joint_b_valid],
            bw[1][joint_b_valid]
        ], axis=1)

        output[f"lag_joint_{duration}"] = (
            minimum_distance(joint_a, joint_b)
        )

    return output


# Pick validation records with two eligible windows.
groups = df.loc[
    (df.split == "validation") & eligible
].groupby("record_id").groups

record_ids = np.array([
    r for r, ids in groups.items()
    if len(ids) >= 2
])

rng4.shuffle(record_ids)
record_ids = record_ids[:60]

rows = []

def collect(scenario, label, record_id,
            ref, candidate, rm, cm):
    rows.append({
        **lag_features(ref, candidate, rm, cm),
        "scenario": scenario,
        "label": label,
        "record_id": record_id
    })


attacks4 = [
    ("shifted_0p5_both", 125, 250, 1800, [0, 1]),
    ("shifted_1s_both", 250, 350, 1500, [0, 1]),
    ("shifted_3s_both", 750, 1200, 150, [0, 1]),
    ("shifted_1s_ch0", 250, 350, 1500, [0]),
    ("shifted_1s_ch1", 250, 350, 1500, [1])
]

for record_id in record_ids:
    ids = np.asarray(list(groups[record_id]), dtype=int)
    i, j = rng4.choice(ids, size=2, replace=False)

    ref, current = X[i], X[j]
    rm, cm = MASK[i], MASK[j]

    # Negative example: genuine different event.
    collect(
        "genuine", 0, record_id,
        ref, current, rm, cm
    )

    # Negative example: genuine with mild noise/drift.
    std = np.array([
        max(np.std(current[ch, cm[ch]]), 1e-4)
        for ch in range(2)
    ])[:, None]

    drift = np.linspace(-1, 1, 2500)[None, :]

    changed = (
        current * rng4.uniform(0.97, 1.03, (2, 1))
        + std * 0.02 * drift
        + rng4.normal(size=current.shape) * std * 0.01
    ).astype(np.float32)

    collect(
        "genuine_mild_drift", 0, record_id,
        ref, changed, rm, cm
    )

    # Shifted replay simulations.
    for name, length, src, dst, channels in attacks4:
        altered = current.copy()
        altered_mask = cm.copy()

        for ch in channels:
            altered[ch, dst:dst+length] = (
                ref[ch, src:src+length]
            )
            altered_mask[ch, dst:dst+length] = (
                rm[ch, src:src+length]
            )

        collect(
            name, 1, record_id,
            ref, altered, rm, altered_mask
        )

probe = pd.DataFrame(rows)

feature_names = [
    c for c in probe.columns if c.startswith("lag_")
]

print("=== LAG-AWARE FEATURE DIAGNOSTIC ===")
print("Records:", len(record_ids))
print("Examples:", len(probe))
print("New features:", len(feature_names))

print("\n=== MEDIAN MATCHING ERRORS ===")
print(
    probe.groupby("scenario")[
        ["lag_ch0_0p5s",
         "lag_ch1_0p5s",
         "lag_joint_0p5s",
         "lag_joint_1s",
         "lag_joint_3s"]
    ].median().round(4).to_string()
)

print("\n=== MISSING-FEATURE FRACTION ===")
print(
    probe[feature_names].isna().mean()
    .round(3).to_string()
)

print("\nSmaller error = closer archived match.")
print("No classifier accuracy is calculated in this step.")
print("STEP 10 COMPLETE")


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

rng5 = np.random.default_rng(1107)

# Use original TRAIN records for feature development.
good = (
    df["split"].eq("train").to_numpy()
    & eligible
)

groups5 = df.loc[good].groupby("record_id").groups

records5 = np.array([
    r for r, ids in groups5.items()
    if len(ids) >= 2
])

rng5.shuffle(records5)
records5 = records5[:90]

results5 = []

def record_case(name, ref, candidate, rm, cm, rec):
    features = lag_features(ref, candidate, rm, cm)
    results5.append({
        "scenario": name,
        "record_id": rec,
        **features
    })


def offgrid_replay(ref, current, rm, cm,
                   length, channels, noisy=False):

    # Random source and destination positions.
    # Ensure offsets differ modulo 5 to test
    # downsampling phase misalignment.
    for _ in range(300):
        src = int(rng5.integers(0, 2501-length))
        dst = int(rng5.integers(0, 2501-length))

        if abs(src-dst) < length:
            continue

        if (src-dst) % 5 == 0:
            continue

        ok = all(
            rm[ch, src:src+length].mean() >= 0.98
            and cm[ch, dst:dst+length].mean() >= 0.98
            for ch in channels
        )

        if ok:
            break
    else:
        return None

    output = current.copy()
    output_mask = cm.copy()

    for ch in channels:
        segment = ref[ch, src:src+length].copy()

        if noisy:
            scale = max(np.std(ref[ch, rm[ch]]), 1e-4)
            segment = (
                1.02 * segment
                + rng5.normal(
                    0, 0.02 * scale, length
                )
            ).astype(np.float32)

        output[ch, dst:dst+length] = segment
        output_mask[ch, dst:dst+length] = (
            rm[ch, src:src+length]
        )

    return output, output_mask


# Scenarios to test
scenarios5 = [
    ("offgrid_0p5_both", 125, [0, 1], False),
    ("offgrid_1s_both", 250, [0, 1], False),
    ("offgrid_1s_noisy", 250, [0, 1], True),
    ("offgrid_3s_both", 750, [0, 1], False),
    ("offgrid_1s_ch0", 250, [0], False),
    ("offgrid_1s_ch1", 250, [1], False),
]


for rec in records5:
    ids = np.asarray(list(groups5[rec]), dtype=int)
    i, j = rng5.choice(ids, size=2, replace=False)

    ref, current = X[i], X[j]
    rm, cm = MASK[i], MASK[j]

    # Genuine comparison
    record_case(
        "genuine", ref, current, rm, cm, rec
    )

    # Genuine with realistic mild noise/drift
    scale = np.array([
        max(np.std(current[ch, cm[ch]]), 1e-4)
        for ch in range(2)
    ])[:, None]

    drift = np.linspace(-1, 1, 2500)[None, :]

    genuine_modified = (
        current * rng5.uniform(0.97, 1.03, (2, 1))
        + 0.02 * scale * drift
        + rng5.normal(size=current.shape) * 0.01 * scale
    ).astype(np.float32)

    record_case(
        "genuine_drift",
        ref,
        genuine_modified,
        rm,
        cm,
        rec
    )

    # Arbitrary-offset simulated replay attacks
    for name, length, channels, noisy in scenarios5:

        result = offgrid_replay(
            ref, current, rm, cm,
            length, channels, noisy
        )

        if result is None:
            continue

        modified, modified_mask = result

        record_case(
            name,
            ref,
            modified,
            rm,
            modified_mask,
            rec
        )


phase_probe = pd.DataFrame(results5)

cols = [
    "lag_ch0_0p5s",
    "lag_ch1_0p5s",
    "lag_joint_0p5s",
    "lag_joint_1s",
    "lag_joint_3s"
]

print("=== ARBITRARY-OFFSET DIAGNOSTIC ===")
print("Records:", len(records5))
print("Examples:", len(phase_probe))

print("\n=== SCENARIO COUNTS ===")
print(
    phase_probe.scenario.value_counts().to_string()
)

print("\n=== MEDIAN MATCHING ERRORS ===")
print(
    phase_probe.groupby("scenario")[cols]
    .median().round(4).to_string()
)

print("\n=== MISSING FEATURE COUNTS ===")
print(
    phase_probe[cols].isna().sum().to_string()
)

# Save results to Google Drive
OUT = Path("/content/drive/MyDrive/SentinelWard-work/m4")
OUT.mkdir(parents=True, exist_ok=True)

phase_probe.to_csv(
    OUT / "v3_offgrid_feature_diagnostic.csv",
    index=False
)

print("\nResults saved successfully!")
print("STEP 11 COMPLETE")


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from numpy.lib.stride_tricks import sliding_window_view

rng6 = np.random.default_rng(1206)

def phase_invariant_features(ref, candidate, rm, cm):
    """
    Match archived signal templates against every incoming
    sample offset, including offsets outside the 50 Hz grid.

    Template start spacing: 25 original samples.
    Candidate start spacing: 1 original sample.
    Samples within templates: every 5th original sample.
    """
    means = np.array([
        ref[ch, rm[ch]].mean() for ch in range(2)
    ])[:, None]

    stds = np.array([
        max(ref[ch, rm[ch]].std(), 1e-4)
        for ch in range(2)
    ])[:, None]

    a = (ref - means) / stds
    b = (candidate - means) / stds

    def minimum_rmse(A, B):
        if len(A) == 0 or len(B) == 0:
            return np.nan

        A = np.ascontiguousarray(A, dtype=np.float64)
        B = np.ascontiguousarray(B, dtype=np.float64)

        aa = np.sum(A*A, axis=1)[:, None]
        bb = np.sum(B*B, axis=1)[None, :]

        distances = np.maximum(
            (aa + bb - 2*(A @ B.T)) / A.shape[1],
            0.0
        )
        return float(np.sqrt(distances.min()))

    result = {}

    # Use shorter matching templates than the attack
    # itself so arbitrary offsets can be recovered.
    for length in [100, 200]:
        name = f"{length/250:.1f}s"

        R = sliding_window_view(
            a, length, axis=1
        )[:, ::25, ::5]

        C = sliding_window_view(
            b, length, axis=1
        )[:, :, ::5]

        RV = sliding_window_view(
            rm, length, axis=1
        )[:, ::25, ::5].all(axis=2)

        CV = sliding_window_view(
            cm, length, axis=1
        )[:, :, ::5].all(axis=2)

        for ch in [0, 1]:
            result[f"ch{ch}_{name}"] = minimum_rmse(
                R[ch][RV[ch]],
                C[ch][CV[ch]]
            )

        # Joint comparison at the same two positions
        rkeep = RV.all(axis=0)
        ckeep = CV.all(axis=0)

        joint_r = np.concatenate(
            [R[0, rkeep], R[1, rkeep]], axis=1
        )
        joint_c = np.concatenate(
            [C[0, ckeep], C[1, ckeep]], axis=1
        )

        result[f"joint_{name}"] = minimum_rmse(
            joint_r, joint_c
        )

    return result


# Sanity check: a copied waveform with arbitrary offsets
# must be recoverable.
test_rng = np.random.default_rng(99)
test_ref = test_rng.normal(size=(2, 2500)).astype("float32")
test_cur = test_rng.normal(size=(2, 2500)).astype("float32")

test_cur[:, 1503:1628] = test_ref[:, 351:476]
test_mask = np.ones_like(test_ref, dtype=bool)

sanity = phase_invariant_features(
    test_ref, test_cur, test_mask, test_mask
)

print("=== MATCHING SANITY CHECK ===")
print("Joint 0.4s error:", sanity["joint_0.4s"])

assert sanity["joint_0.4s"] < 0.001, (
    "Matching sanity check failed!"
)

# Use original training records for development.
good6 = (
    df["split"].eq("train").to_numpy()
    & eligible
)

groups6 = df.loc[good6].groupby("record_id").groups

records6 = np.array([
    r for r, ids in groups6.items()
    if len(ids) >= 2
])

rng6.shuffle(records6)
records6 = records6[:60]

results6 = []

def add_case6(name, rec, ref, candidate, rm, cm):
    result = phase_invariant_features(
        ref, candidate, rm, cm
    )
    results6.append({
        "scenario": name,
        "record_id": rec,
        **result
    })

# Reuse the offgrid_replay simulator from Cell 11.
scenarios6 = [
    ("offgrid_0p5_both", 125, [0, 1], False),
    ("offgrid_1s_both", 250, [0, 1], False),
    ("offgrid_1s_noisy", 250, [0, 1], True),
    ("offgrid_3s_both", 750, [0, 1], False),
    ("offgrid_1s_ch0", 250, [0], False),
    ("offgrid_1s_ch1", 250, [1], False),
]

for rec in records6:
    ids = np.asarray(list(groups6[rec]), dtype=int)
    i, j = rng6.choice(ids, size=2, replace=False)

    ref, current = X[i], X[j]
    rm, cm = MASK[i], MASK[j]

    add_case6("genuine", rec, ref, current, rm, cm)

    # Modified genuine recording.
    scale = np.array([
        max(np.std(current[ch, cm[ch]]), 1e-4)
        for ch in range(2)
    ])[:, None]

    drift = np.linspace(-1, 1, 2500)[None, :]

    altered_genuine = (
        current * rng6.uniform(0.97, 1.03, (2, 1))
        + 0.02 * scale * drift
        + rng6.normal(size=current.shape) * 0.01 * scale
    ).astype(np.float32)

    add_case6(
        "genuine_drift", rec,
        ref, altered_genuine, rm, cm
    )

    # Shifted synthetic replay cases.
    for name, length, channels, noisy in scenarios6:
        attack = offgrid_replay(
            ref, current, rm, cm,
            length, channels, noisy
        )

        if attack is None:
            continue

        modified, modified_mask = attack

        add_case6(
            name, rec,
            ref, modified, rm, modified_mask
        )

probe12 = pd.DataFrame(results6)

cols12 = [
    "ch0_0.4s", "ch1_0.4s", "joint_0.4s",
    "ch0_0.8s", "ch1_0.8s", "joint_0.8s"
]

print("\n=== PHASE-AWARE DIAGNOSTIC ===")
print("Records:", len(records6))
print("Examples:", len(probe12))

print("\n=== SCENARIO COUNTS ===")
print(probe12.scenario.value_counts().to_string())

print("\n=== MEDIAN MATCHING ERRORS ===")
print(
    probe12.groupby("scenario")[cols12]
    .median().round(5).to_string()
)

print("\n=== JOINT 0.4s ERROR DISTRIBUTION ===")
distribution = probe12.groupby("scenario")[
    "joint_0.4s"
].agg(
    q10=lambda x: x.quantile(0.10),
    median="median",
    q90=lambda x: x.quantile(0.90)
)
print(distribution.round(5).to_string())

print("\n=== MISSING FEATURE COUNTS ===")
print(probe12[cols12].isna().sum().to_string())

OUT = Path("/content/drive/MyDrive/SentinelWard-work/m4")
OUT.mkdir(parents=True, exist_ok=True)

probe12.to_csv(
    OUT / "v3_phase_aware_diagnostic.csv",
    index=False
)

print("\nSTEP 12 COMPLETE")


In [ ]:

import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score
)

# Reuse the Cell 12 diagnostic dataset.
data13 = probe12.copy()
features13 = [
    "ch0_0.4s", "ch1_0.4s", "joint_0.4s",
    "ch0_0.8s", "ch1_0.8s", "joint_0.8s"
]

data13["label"] = (
    ~data13["scenario"].isin(["genuine", "genuine_drift"])
).astype(int)

assert len(data13) > 0
assert np.isfinite(data13[features13].to_numpy()).all()

# Group-separated 60/20/20 split.
rng13 = np.random.default_rng(1313)
record_ids13 = data13.record_id.unique().copy()
rng13.shuffle(record_ids13)

n13 = len(record_ids13)
fit_ids = set(record_ids13[:int(n13 * 0.60)])
dev_ids = set(record_ids13[
    int(n13 * 0.60):int(n13 * 0.80)
])
test_ids = set(record_ids13[int(n13 * 0.80):])

assert not (fit_ids & dev_ids)
assert not (fit_ids & test_ids)
assert not (dev_ids & test_ids)

fit13 = data13[data13.record_id.isin(fit_ids)].copy()
dev13 = data13[data13.record_id.isin(dev_ids)].copy()
test13 = data13[data13.record_id.isin(test_ids)].copy()

print("=== V3 DATA PARTITIONS ===")
for name, part in [
    ("Fit", fit13), ("Development", dev13), ("Test", test13)
]:
    print(name, "records:", part.record_id.nunique(),
          "examples:", len(part))

# Train V3 Random Forest.
model4_v3 = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

model4_v3.fit(fit13[features13], fit13.label)

# Set threshold using development genuine examples only.
dev13["score"] = model4_v3.predict_proba(
    dev13[features13]
)[:, 1]

neg_scores13 = dev13.loc[
    dev13.label == 0, "score"
].to_numpy()

threshold13 = float(
    np.nextafter(np.quantile(neg_scores13, 0.99), np.inf)
)

print("\nFrozen V3 threshold:", round(threshold13, 5))
print("Note: Small development set; FPR target is not guaranteed.")

def evaluate13(part, name):
    result = part.copy()
    result["score"] = model4_v3.predict_proba(
        result[features13]
    )[:, 1]
    result["prediction"] = (
        result.score >= threshold13
    ).astype(int)

    print(f"\n=== {name} ===")
    print("Confusion matrix [TN FP; FN TP]:")
    print(confusion_matrix(
        result.label, result.prediction, labels=[0, 1]
    ))
    print(classification_report(
        result.label, result.prediction,
        labels=[0, 1],
        target_names=["Genuine", "Simulated replay"],
        zero_division=0
    ))
    print("ROC-AUC:", round(roc_auc_score(
        result.label, result.score
    ), 4))

    summary = result.groupby(
        ["scenario", "label"], as_index=False
    ).agg(
        cases=("prediction", "size"),
        flagged=("prediction", "sum"),
        mean_score=("score", "mean")
    )
    summary["flagged_percent"] = (
        100 * summary.flagged / summary.cases
    ).round(2)

    print("\nScenario results:")
    print(summary[
        ["scenario", "cases", "flagged_percent", "mean_score"]
    ].to_string(index=False))

    return result, summary

dev_results13, _ = evaluate13(dev13, "V3 DEVELOPMENT")
test_results13, summary13 = evaluate13(
    test13, "V3 INTERNAL TEST"
)

importance13 = pd.Series(
    model4_v3.feature_importances_,
    index=features13
).sort_values(ascending=False)

print("\n=== FEATURE IMPORTANCE ===")
print(importance13.to_string())

# Save without overwriting V1 or V2.
OUT = Path("/content/drive/MyDrive/SentinelWard-work/m4")
OUT.mkdir(parents=True, exist_ok=True)

joblib.dump({
    "model": model4_v3,
    "feature_columns": features13,
    "threshold": threshold13,
    "version": "v3_exploratory",
    "sampling_rate_hz": 250,
    "matcher": "phase_invariant_features_cell12",
    "evaluation": "synthetic_record_grouped",
    "patient_disjoint_verified": False
}, OUT / "model4_v3_exploratory.joblib")

test_results13.to_csv(
    OUT / "v3_internal_test_predictions.csv",
    index=False
)
summary13.to_csv(
    OUT / "v3_internal_test_summary.csv",
    index=False
)

print("\nMODEL 4 V3 PROTOTYPE SAVED")
print("STEP 13 COMPLETE")


In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

# Use predictions already generated in Cell 13.
dev14 = dev_results13.copy()
test14 = test_results13.copy()

print("=== V3 SCORE DISTRIBUTIONS ===")

stats14 = test14.groupby("scenario")["score"].agg(
    count="size",
    minimum="min",
    q10=lambda s: s.quantile(0.10),
    median="median",
    q90=lambda s: s.quantile(0.90),
    maximum="max"
)

print(stats14.round(4).to_string())

print("\n=== MISSED REPLAY BREAKDOWN ===")

missed14 = test14[
    (test14.label == 1) &
    (test14.prediction == 0)
]

print("Total missed attacks:", len(missed14))
print(missed14["scenario"].value_counts().to_string())

print("\n=== FALSE POSITIVE BREAKDOWN ===")

false14 = test14[
    (test14.label == 0) &
    (test14.prediction == 1)
]

print("False positives:", len(false14))
print(false14["scenario"].value_counts().to_string())

print("\n=== DEVELOPMENT THRESHOLD ANALYSIS ===")

# Explore threshold sensitivity on development data ONLY.
for t in [0.30, 0.50, 0.70, 0.80, 0.90, threshold13]:
    pred = (dev14.score >= t).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        dev14.label, pred, labels=[0, 1]
    ).ravel()

    recall = tp / max(tp + fn, 1)
    fpr = fp / max(fp + tn, 1)

    noisy = dev14.scenario == "offgrid_1s_noisy"
    noisy_recall = float(
        (dev14.loc[noisy, "score"] >= t).mean()
    )

    print(
        f"Threshold={t:.4f} | "
        f"Replay recall={recall:.3f} | "
        f"Noisy recall={noisy_recall:.3f} | "
        f"FPR={fpr:.3f}"
    )

print("\n=== FEATURE VALUES OF MISSED ATTACKS ===")

if len(missed14):
    display(
        missed14[
            ["scenario", "score"] + features13
        ].round(4)
    )

print("\nSTEP 14 COMPLETE")


In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score
)

# Use existing Cell 13 results.
dev15 = dev_results13.copy()
test15 = test_results13.copy()

# A smaller normalized matching error means a closer match.
# Take the smallest error across both channels and durations.
def matching_error(frame):
    return frame[
        ["ch0_0.4s", "ch1_0.4s",
         "ch0_0.8s", "ch1_0.8s"]
    ].min(axis=1).to_numpy()

dev_error = matching_error(dev15)
test_error = matching_error(test15)

# Calibrate the rule on development genuine cases ONLY.
genuine_dev = dev_error[dev15.label.to_numpy() == 0]

# Strictly below the smallest genuine development error.
# This deliberately avoids development false positives.
rule_threshold = float(np.min(genuine_dev))

rule_pred = (test_error < rule_threshold).astype(int)

print("=== SIMPLE MATCHING RULE ===")
print("Rule threshold:", round(rule_threshold, 6))
print("Confusion matrix:")
print(confusion_matrix(
    test15.label, rule_pred, labels=[0, 1]
))
print(classification_report(
    test15.label, rule_pred,
    labels=[0, 1],
    target_names=["Genuine", "Replay"],
    zero_division=0
))

# Compare scenario-level detection.
comparison = test15[
    ["record_id", "scenario", "label"]
].copy()

comparison["rf_prediction"] = test15.prediction.to_numpy()
comparison["rule_prediction"] = rule_pred

report15 = comparison.groupby("scenario").agg(
    cases=("label", "size"),
    rf_flagged=("rf_prediction", "mean"),
    rule_flagged=("rule_prediction", "mean")
)

report15[["rf_flagged", "rule_flagged"]] *= 100

print("\n=== RANDOM FOREST VS MATCHING RULE ===")
print(report15.round(2).to_string())

# Inspect genuine examples most likely to be falsely flagged.
print("\n=== HARDEST GENUINE EXAMPLES ===")
hard = test15[test15.label == 0].copy()
hard["matching_error"] = matching_error(hard)

print(hard[
    ["record_id", "scenario", "score", "matching_error"]
].sort_values("matching_error").head(8).to_string(index=False))

print("\nSTEP 15 COMPLETE")


In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

dev16 = dev_results13.copy()
test16 = test_results13.copy()

# Evaluate individual and joint-channel matching.
def get_errors(frame):
    return pd.DataFrame({
        "single_min": frame[
            ["ch0_0.4s", "ch1_0.4s",
             "ch0_0.8s", "ch1_0.8s"]
        ].min(axis=1),
        "joint_min": frame[
            ["joint_0.4s", "joint_0.8s"]
        ].min(axis=1)
    }, index=frame.index)

dev_errors = get_errors(dev16)

print("=== GENUINE DEVELOPMENT MATCHES ===")
genuine_dev = dev16[dev16.label == 0].copy()
genuine_dev["single_min"] = dev_errors.loc[
    genuine_dev.index, "single_min"
]
genuine_dev["joint_min"] = dev_errors.loc[
    genuine_dev.index, "joint_min"
]

print(genuine_dev[
    ["record_id", "scenario", "single_min", "joint_min"]
].sort_values("single_min").head(12).to_string(index=False))

print("\n=== ZERO-ERROR GENUINE MATCHES ===")
print("Single-channel zero matches:",
      int((genuine_dev.single_min < 1e-6).sum()))
print("Joint-channel zero matches:",
      int((genuine_dev.joint_min < 1e-6).sum()))

# Threshold sweep: development set only.
print("\n=== DEVELOPMENT RULE COMPARISON ===")

for strategy in ["single_min", "joint_min"]:
    errors = dev_errors[strategy].to_numpy()
    truth = dev16.label.to_numpy()

    print("\nStrategy:", strategy)

    for threshold in [0.005, 0.01, 0.02, 0.03, 0.05, 0.08]:
        pred = (errors <= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            truth, pred, labels=[0, 1]
        ).ravel()

        recall = tp / max(tp + fn, 1)
        fpr = fp / max(fp + tn, 1)

        noisy = dev16.scenario.eq("offgrid_1s_noisy")
        noisy_recall = pred[noisy.to_numpy()].mean()

        print(
            f"t={threshold:.3f} | "
            f"Recall={recall:.3f} | "
            f"Noisy recall={noisy_recall:.3f} | "
            f"FPR={fpr:.3f}"
        )

print("\nSTEP 16 COMPLETE")


In [ ]:

import numpy as np
import pandas as pd

TARGET = "0719c6"

# Find eligible windows belonging to the suspicious record.
ids17 = np.flatnonzero(
    (df["record_id"].to_numpy() == TARGET) & eligible
)

print("=== SUSPICIOUS RECORD ===")
print("Record ID:", TARGET)
print("Eligible windows:", len(ids17))

print("\n=== EVENT INFORMATION ===")
print(df.iloc[ids17][
    ["event_id", "record_id", "split", "vt_label"]
].to_string(index=True))

print("\n=== PAIRWISE SIGNAL COMPARISON ===")

for a_pos in range(len(ids17)):
    for b_pos in range(a_pos + 1, len(ids17)):
        i, j = ids17[a_pos], ids17[b_pos]

        a, b = X[i], X[j]
        ma, mb = MASK[i], MASK[j]

        # Compare entire windows.
        common = ma & mb
        exact_fraction = np.mean(
            np.isclose(a[common], b[common],
                       rtol=0, atol=1e-7)
        ) if common.any() else np.nan

        # Existing lag-aware features.
        lag = phase_invariant_features(
            a, b, ma, mb
        )

        print("\nEvents:", df.iloc[i].event_id,
              "vs", df.iloc[j].event_id)

        print("Aligned exact fraction:",
              round(float(exact_fraction), 5))

        print("Single ch0 0.4s:",
              round(lag["ch0_0.4s"], 6))
        print("Single ch1 0.4s:",
              round(lag["ch1_0.4s"], 6))
        print("Joint 0.4s:",
              round(lag["joint_0.4s"], 6))

print("\n=== ORIGINAL HEADER METADATA ===")
manifest17 = manifest[
    manifest["record_id"].astype(str) == TARGET
]

print(manifest17[
    ["event_id", "sampling_rate_hz",
     "samples", "channel_names",
     "preferred_ecg", "has_pleth_header"]
].to_string(index=False))

print("\nSTEP 17 COMPLETE")


In [ ]:

import cloudpickle
import numpy as np
import sklearn
import pandas as pd
import json
from pathlib import Path

OUT = Path("/content/drive/MyDrive/SentinelWard-work/m4/final")
OUT.mkdir(parents=True, exist_ok=True)

# Store the Python feature extractor with the fitted model.
bundle = {
    "name": "SentinelWard Model 4",
    "version": "3.0-research",
    "model": model4_v3,
    "feature_extractor": phase_invariant_features,
    "feature_columns": list(features13),
    "threshold": float(threshold13),
    "sample_rate_hz": 250,
    "window_samples": 2500,
    "minimum_valid_fraction_per_channel": 0.90,
    "channels": 2,
    "channel_order_verified": False,
    "sklearn_version": sklearn.__version__,
    "purpose": "Offline synthetic replay similarity detection",
    "deployment_status": "research_only"
}

path = OUT / "sentinelward_model4_v3.pkl"

with open(path, "wb") as f:
    cloudpickle.dump(bundle, f)

# Verify the complete package by reloading it.
# Only load files created by your own trusted environment.
with open(path, "rb") as f:
    loaded = cloudpickle.load(f)

# Use one known evaluation record to check inference.
# This checks packaging consistency, not model accuracy.
sample = test13.iloc[[0]]
saved_features = loaded["feature_columns"]

original_score = float(
    model4_v3.predict_proba(
        sample[saved_features]
    )[0, 1]
)

restored_score = float(
    loaded["model"].predict_proba(
        sample[saved_features]
    )[0, 1]
)

assert np.isclose(original_score, restored_score)

# Test that the exported extractor works on raw windows.
record = str(sample.iloc[0]["record_id"])
indices = np.flatnonzero(
    (df["record_id"].to_numpy() == record) & eligible
)

assert len(indices) >= 2, "Need two eligible windows"

i, j = indices[:2]

extracted = loaded["feature_extractor"](
    X[i], X[j], MASK[i], MASK[j]
)

assert all(
    k in extracted and np.isfinite(extracted[k])
    for k in saved_features
)

raw_features = pd.DataFrame(
    [{k: extracted[k] for k in saved_features}]
)

raw_score = float(
    loaded["model"].predict_proba(raw_features)[0, 1]
)

raw_flag = int(raw_score >= loaded["threshold"])

print("=== FINAL PACKAGE VERIFICATION ===")
print("Model:", loaded["name"])
print("Version:", loaded["version"])
print("Threshold:", loaded["threshold"])
print("Feature count:", len(saved_features))
print("Serialization test: PASSED")
print("Feature extraction test: PASSED")
print("Raw-window inference test: PASSED")
print("Example score:", round(raw_score, 6))
print("Example flag:", raw_flag)
print("Package:", path)
print("Size MB:", round(path.stat().st_size / 1e6, 3))

# Manifest for IBM project integration
manifest_final = {
    "model": loaded["name"],
    "version": loaded["version"],
    "artifact": path.name,
    "sampling_rate": 250,
    "window_samples": 2500,
    "features": saved_features,
    "threshold": loaded["threshold"],
    "model_type": "RandomForest",
    "output": {
        "0": "No replay detected",
        "1": "Possible replay similarity"
    },
    "limitations": [
        "Research prototype",
        "Requires two aligned-format 10s windows",
        "Reference window must be available",
        "Does not prove unauthorized replay",
        "Not a replacement for authenticated freshness checks"
    ]
}

with open(OUT / "deployment_manifest.json", "w") as f:
    json.dump(manifest_final, f, indent=2)

print("\nMODEL 4 PACKAGE COMPLETE")
print("CELL 18 COMPLETE")
